### ***EMPLOYEE DATA SIMILARITY SEARCH***

- ***Create a collection for employee data and store related data in a Chroma DB collection.***
- ***Generate numeric representations for specific key-value pairs of the employee dataset and convert them into vector embeddings.***
- ***Store related data in a Chroma DB collection.***

#### ***Step 1***: ***Importing the required libraries***

In [1]:
import chromadb # chromadb is used to interact with the Chroma DB database,
from chromadb.utils import embedding_functions # embedding_functions is used to define the embedding model

#### ***Step 2: Define the embedding function***

In [2]:
# Define the embedding function using SentenceTransformers
# This function will be used to generate embeddings (vector representations) for the data

ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

In [3]:
print(ef.models)
print(ef.model_name)
print(ef._model)

{'all-MiniLM-L6-v2': SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)}
all-MiniLM-L6-v2
SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)


#### ***Step 3: Create an instance of Chroma Client***

In [4]:
client = chromadb.Client() # Creating an instance of ChromaClient to establish a connection with the Chroma database

#### ***Step 4: Create Collection Name***

In [5]:
collection_name = "employee_collection" # Define the name of the collection where employee data will be stored

#### ***Step 5: Create Collection***

In [6]:
collection = client.create_collection(
    name=collection_name, # Specifying the name of the collection to be created
    metadata={"description": "A collection for storing employee data"}, # Adding metadata to describe the collection
    configuration={
        "hnsw": {"space": "cosine"}, # Configuring the collection with cosine distance and embedding function
        "embedding_function": ef # Embedding function
    }
)

print(f"Collection created: {collection.name}")

Collection created: employee_collection


#### ***Step 6: Construct the input data***

In [7]:
# Each dictionary represents an individual employee with comprehensive information
employees = [{"id": "employee_1","name": "John Doe","experience": 5,"department": "Engineering","role": "Software Engineer","skills": "Python, JavaScript, React, Node.js, databases","location": "New York","employment_type": "Full-time"},
            {"id": "employee_2","name": "Jane Smith","experience": 8,"department": "Marketing","role": "Marketing Manager","skills": "Digital marketing, SEO, content strategy, analytics, social media","location": "Los Angeles","employment_type": "Full-time"},
            {"id": "employee_3","name": "Alice Johnson","experience": 3,"department": "HR","role": "HR Coordinator","skills": "Recruitment, employee relations, HR policies, training programs","location": "Chicago","employment_type": "Full-time"},
            {"id": "employee_4","name": "Michael Brown","experience": 12,"department": "Engineering","role": "Senior Software Engineer","skills": "Java, Spring Boot, microservices, cloud architecture, DevOps","location": "San Francisco","employment_type": "Full-time"},
            {"id": "employee_5","name": "Emily Wilson","experience": 2,"department": "Marketing","role": "Marketing Assistant","skills": "Content creation, email marketing, market research, social media management","location": "Austin","employment_type": "Part-time"},
            {"id": "employee_6","name": "David Lee","experience": 15,"department": "Engineering","role": "Engineering Manager","skills": "Team leadership, project management, software architecture, mentoring","location": "Seattle","employment_type": "Full-time"},
            {"id": "employee_7","name": "Sarah Clark","experience": 8,"department": "HR","role": "HR Manager","skills": "Performance management, compensation planning, policy development, conflict resolution","location": "Boston","employment_type": "Full-time"},
            {"id": "employee_8","name": "Chris Evans","experience": 20,"department": "Engineering","role": "Senior Architect","skills": "System design, distributed systems, cloud platforms, technical strategy","location": "New York","employment_type": "Full-time"},
            {"id": "employee_9","name": "Jessica Taylor","experience": 4,"department": "Marketing","role": "Marketing Specialist","skills": "Brand management, advertising campaigns, customer analytics, creative strategy","location": "Miami","employment_type": "Full-time"},
            {"id": "employee_10","name": "Alex Rodriguez","experience": 18,"department": "Engineering","role": "Lead Software Engineer","skills": "Full-stack development, React, Python, machine learning, data science","location": "Denver","employment_type": "Full-time"},
            {"id": "employee_11","name": "Hannah White","experience": 6,"department": "HR","role": "HR Business Partner","skills": "Strategic HR, organizational development, change management, employee engagement","location": "Portland","employment_type": "Full-time"},
            {"id": "employee_12","name": "Kevin Martinez","experience": 10,"department": "Engineering","role": "DevOps Engineer","skills": "Docker, Kubernetes, AWS, CI/CD pipelines, infrastructure automation","location": "Phoenix","employment_type": "Full-time"},
            {"id": "employee_13","name": "Rachel Brown","experience": 7,"department": "Marketing","role": "Marketing Director","skills": "Strategic marketing, team leadership, budget management, campaign optimization","location": "Atlanta","employment_type": "Full-time"},
            {"id": "employee_14","name": "Matthew Garcia","experience": 3,"department": "Engineering","role": "Junior Software Engineer","skills": "JavaScript, HTML/CSS, basic backend development, learning frameworks","location": "Dallas","employment_type": "Full-time"},
            {"id": "employee_15","name": "Olivia Moore","experience": 12,"department": "Engineering","role": "Principal Engineer","skills": "Technical leadership, system architecture, performance optimization, mentoring","location": "San Francisco","employment_type": "Full-time"},
    ]

#### ***Step 7: Create Comprehensive text document for each employee***

In [8]:
# These documents will be used for similarity search based on skills, roles, and experience
employee_documents = []
for employee in employees:
    document = f"{employee['role']} with {employee['experience']} years of experience in {employee['department']}. "
    document += f"Skills: {employee['skills']}. Located in {employee['location']}. "
    document += f"Employment type: {employee['employment_type']}."
    employee_documents.append(document)

In [9]:
employee_documents[0]

'Software Engineer with 5 years of experience in Engineering. Skills: Python, JavaScript, React, Node.js, databases. Located in New York. Employment type: Full-time.'

#### ***Step 8: Add documents to vector store***

In [10]:
# The 'add' method inserts or updates data into the specified collection
collection.add(
    # Extracting employee IDs to be used as unique identifiers for each record
    ids=[employee["id"] for employee in employees],
    # Using the comprehensive text documents we created
    documents=employee_documents,
    # Adding comprehensive metadata for filtering and search
    metadatas=[{
        "name": employee["name"],
        "department": employee["department"],
        "role": employee["role"],
        "experience": employee["experience"],
        "location": employee["location"],
        "employment_type": employee["employment_type"]
    } for employee in employees]
)

In [11]:
# Check the total count
print(f"Total items in collection: {collection.count()}")

Total items in collection: 15


In [12]:
# See the first few records to ensure metadata and text look right
results = collection.peek()
print(results["ids"])
print(results["metadatas"][0]) # Check the first employee's metadata

['employee_1', 'employee_2', 'employee_3', 'employee_4', 'employee_5', 'employee_6', 'employee_7', 'employee_8', 'employee_9', 'employee_10']
{'role': 'Software Engineer', 'location': 'New York', 'department': 'Engineering', 'experience': 5, 'name': 'John Doe', 'employment_type': 'Full-time'}


#### ***Step 9: Check the total count***

In [13]:
all_items = collection.get()
# LOG THE RETRIEVED ITEMS TO THE CONSOLE FOR INSPECTION
print("Collection contents:")
print(f"Number of documents: {len(all_items['documents'])}")

Collection contents:
Number of documents: 15


#### ***Step 10 - Advance Query***

***Step 10.1 - Similarity Search Examples***

In [14]:
# search for python developers

print("\n1. Searching for Python developers:")

query_text = "Python developer with web development experience"

results = collection.query(query_texts=[query_text],n_results=3) # returns dictionary of ids, documents, distances, metadatas

print(f"Query: '{query_text}'")

for i, (doc_id, document, distance) in enumerate(zip(results['ids'][0], results['documents'][0], results['distances'][0])):
    metadata = results['metadatas'][0][i]
    print(f"  {i+1}. {metadata['name']} ({doc_id}) - Distance: {distance:.4f}")
    print(f"     Role: {metadata['role']}, Department: {metadata['department']}")
    print(f"     Document: {document[:100]}...")


1. Searching for Python developers:
Query: 'Python developer with web development experience'
  1. John Doe (employee_1) - Distance: 0.5156
     Role: Software Engineer, Department: Engineering
     Document: Software Engineer with 5 years of experience in Engineering. Skills: Python, JavaScript, React, Node...
  2. Matthew Garcia (employee_14) - Distance: 0.5724
     Role: Junior Software Engineer, Department: Engineering
     Document: Junior Software Engineer with 3 years of experience in Engineering. Skills: JavaScript, HTML/CSS, ba...
  3. Alex Rodriguez (employee_10) - Distance: 0.5967
     Role: Lead Software Engineer, Department: Engineering
     Document: Lead Software Engineer with 18 years of experience in Engineering. Skills: Full-stack development, R...


***Step 10.2 - Similarity Search Example***

In [15]:
# Example 2: Search for leadership roles
print("\n2. Searching for leadership and management roles:")
query_text = "team leader manager with experience"
results = collection.query(
    query_texts=[query_text],
    n_results=3
)
print(f"Query: '{query_text}'")
for i, (doc_id, document, distance) in enumerate(zip(
    results['ids'][0], results['documents'][0], results['distances'][0]
)):
    metadata = results['metadatas'][0][i]
    print(f"  {i+1}. {metadata['name']} ({doc_id}) - Distance: {distance:.4f}")
    print(f"     Role: {metadata['role']}, Experience: {metadata['experience']} years")


2. Searching for leadership and management roles:
Query: 'team leader manager with experience'
  1. Jane Smith (employee_2) - Distance: 0.5382
     Role: Marketing Manager, Experience: 8 years
  2. Sarah Clark (employee_7) - Distance: 0.5467
     Role: HR Manager, Experience: 8 years
  3. David Lee (employee_6) - Distance: 0.5497
     Role: Engineering Manager, Experience: 15 years


***Step 10.3 - Metadata Filtering Example***

In [16]:
# Example 1: Filter by department
print("\n3. Finding all Engineering employees:")
results = collection.get(
    where={"department": "Engineering"}
)
print(f"Found {len(results['ids'])} Engineering employees:")
for i, doc_id in enumerate(results['ids']):
    metadata = results['metadatas'][i]
    print(f"  - {metadata['name']}: {metadata['role']} ({metadata['experience']} years)")


3. Finding all Engineering employees:
Found 8 Engineering employees:
  - John Doe: Software Engineer (5 years)
  - Michael Brown: Senior Software Engineer (12 years)
  - David Lee: Engineering Manager (15 years)
  - Chris Evans: Senior Architect (20 years)
  - Alex Rodriguez: Lead Software Engineer (18 years)
  - Kevin Martinez: DevOps Engineer (10 years)
  - Matthew Garcia: Junior Software Engineer (3 years)
  - Olivia Moore: Principal Engineer (12 years)


In [17]:
# Example 2: Filter by experience range
print("\n4. Finding employees with 10+ years experience:")
results = collection.get(
    where={"experience": {"$gte": 10}}
)
print(f"Found {len(results['ids'])} senior employees:")
for i, doc_id in enumerate(results['ids']):
    metadata = results['metadatas'][i]
    print(f"  - {metadata['name']}: {metadata['role']} ({metadata['experience']} years)")


4. Finding employees with 10+ years experience:
Found 6 senior employees:
  - Michael Brown: Senior Software Engineer (12 years)
  - David Lee: Engineering Manager (15 years)
  - Chris Evans: Senior Architect (20 years)
  - Alex Rodriguez: Lead Software Engineer (18 years)
  - Kevin Martinez: DevOps Engineer (10 years)
  - Olivia Moore: Principal Engineer (12 years)


In [18]:
# Example 3: Filter by location
print("\n5. Finding employees in California:")
results = collection.get(
    where={"location": {"$in": ["San Francisco", "Los Angeles"]}}
)
print(f"Found {len(results['ids'])} employees in California:")
for i, doc_id in enumerate(results['ids']):
    metadata = results['metadatas'][i]
    print(f"  - {metadata['name']}: {metadata['location']}")


5. Finding employees in California:
Found 3 employees in California:
  - Jane Smith: Los Angeles
  - Michael Brown: San Francisco
  - Olivia Moore: San Francisco


***Step 10.4 - Combined Search: Similarity and Filter***

In [19]:
# Example: Find experienced Python developers in specific locations
print("\n6. Finding senior Python developers in major tech cities:")
query_text = "senior Python developer full-stack"
results = collection.query(
    query_texts=[query_text],
    n_results=5,
    where={
        "$and": [
            {"experience": {"$gte": 8}},
            {"location": {"$in": ["San Francisco", "New York", "Seattle"]}}
        ]
    }
)
print(f"Query: '{query_text}' with filters (8+ years, major tech cities)")
print(f"Found {len(results['ids'][0])} matching employees:")
for i, (doc_id, document, distance) in enumerate(zip(
    results['ids'][0], results['documents'][0], results['distances'][0]
)):
    metadata = results['metadatas'][0][i]
    print(f"  {i+1}. {metadata['name']} ({doc_id}) - Distance: {distance:.4f}")
    print(f"     {metadata['role']} in {metadata['location']} ({metadata['experience']} years)")
    print(f"     Document snippet: {document[:80]}...")


6. Finding senior Python developers in major tech cities:
Query: 'senior Python developer full-stack' with filters (8+ years, major tech cities)
Found 4 matching employees:
  1. Michael Brown (employee_4) - Distance: 0.6726
     Senior Software Engineer in San Francisco (12 years)
     Document snippet: Senior Software Engineer with 12 years of experience in Engineering. Skills: Jav...
  2. Chris Evans (employee_8) - Distance: 0.7537
     Senior Architect in New York (20 years)
     Document snippet: Senior Architect with 20 years of experience in Engineering. Skills: System desi...
  3. David Lee (employee_6) - Distance: 0.8344
     Engineering Manager in Seattle (15 years)
     Document snippet: Engineering Manager with 15 years of experience in Engineering. Skills: Team lea...
  4. Olivia Moore (employee_15) - Distance: 0.8761
     Principal Engineer in San Francisco (12 years)
     Document snippet: Principal Engineer with 12 years of experience in Engineering. Skills: Technical..

***Thanks all for now!!!***